# Agnes AI 图像生成

使用 Agnes Image API 生成图像的 Python 示例

In [21]:
import requests
import json
import os
from IPython.display import display, Image as IPImage

# === 配置 ===
# 方式1: 从环境变量读取（推荐）
# 在终端设置: set AGNES_API_KEY=your_key_here
API_KEY = os.getenv("AGNES_API_KEY")
# print(API_KEY)

# 方式2: 直接填写（不推荐，注意不要提交到Git）
# API_KEY = "YOUR_API_KEY"

if not API_KEY:
    raise ValueError("请设置 AGNES_API_KEY 环境变量或直接填写 API_KEY")

API_URL = "https://apihub.agnes-ai.com/v1/images/generations"

In [ ]:
# === Cell 1: Text-to-Image（文生图）===

payload = {
    "model": "agnes-image-2.1-flash",
    "prompt": "A luminous floating city above a misty canyon at sunrise, cinematic realism",
    "size": "2K",
    "ratio": "16:9",
    "extra_body": {
        "response_format": "url"
    }
}

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# === 检查系统代理设置 ===
for k in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"]:
    if os.environ.get(k):
        print(f"[!] 检测到代理: {k}={os.environ[k]}")

# === 发送请求（绕过系统代理，避免 ProxyError）===
response = requests.post(
    API_URL,
    headers=headers,
    json=payload,
    proxies={"http": None, "https": None}
)
print(f"状态码: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print("\n生成成功！响应数据：")
    print(json.dumps(result, indent=2, ensure_ascii=False))
    
    try:
        image_url = result["data"][0]["url"]
        print(f"\n图像 URL: {image_url}")
        display(IPImage(url=image_url))
    except (KeyError, IndexError):
        print("\n(图像 URL 提取失败，请检查响应结构)")
else:
    print(f"\n请求失败: {response.text}")

In [20]:
# === Cell 2: Image-to-Image（图生图：以参考图片生成新图）===

# 上一轮生成的图片 URL，可替换为你自己的图片 URL
INPUT_IMAGE_URL = "https://platform-outputs.agnes-ai.space/images/t2i/36c953c8a42a422eacdb124e39ddec5d.png"

prompt=f'''
一座充满神秘气息的悬浮城市，装饰着闪闪发光的水晶结构和发出魔法光芒的灯光，悬浮在晨雾弥漫的峡谷之上，沐浴在金色而梦幻的日出光辉中。城市周围环绕着发光的雾气，空气中弥漫着神秘的能量流动，雄伟的飞翼生物在充满活力的天空中翱翔。具备电影般的奇幻写实风格，展现精致而超凡的建筑设计和充满魔力氛围的景象。
'''
#"Transform the scene into a rain-soaked cyberpunk night with neon reflections while preserving the original composition",
payload = {
    "model": "agnes-image-2.1-flash",
    "prompt": prompt,
    "size": "2K",
    "ratio": "16:9",
    "extra_body": {
        "image": [INPUT_IMAGE_URL],
        "response_format": "url"
    }
}

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

print(f"参考图片: {INPUT_IMAGE_URL}")
print(f"Prompt: {payload['prompt']}")
print()

response = requests.post(
    API_URL,
    headers=headers,
    json=payload,
    proxies={"http": None, "https": None}
)
print(f"状态码: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print(json.dumps(result, indent=2, ensure_ascii=False))
    try:
        image_url = result["data"][0]["url"]
        display(IPImage(url=image_url))
    except (KeyError, IndexError):
        print("(图像 URL 提取失败)")
else:
    print(f"请求失败: {response.text}")

参考图片: https://platform-outputs.agnes-ai.space/images/t2i/36c953c8a42a422eacdb124e39ddec5d.png
Prompt: 
一座充满神秘气息的悬浮城市，装饰着闪闪发光的水晶结构和发出魔法光芒的灯光，悬浮在晨雾弥漫的峡谷之上，沐浴在金色而梦幻的日出光辉中。城市周围环绕着发光的雾气，空气中弥漫着神秘的能量流动，雄伟的飞翼生物在充满活力的天空中翱翔。具备电影般的奇幻写实风格，展现精致而超凡的建筑设计和充满魔力氛围的景象。


状态码: 200
{
  "data": [
    {
      "url": "https://platform-outputs.agnes-ai.space/images/i2i/task_8yPE915Fce7pDnIhCP8PoBwaw8U5JlTB/output.png",
      "b64_json": "",
      "revised_prompt": ""
    }
  ],
  "created": 1784632097,
  "task_id": "task_8yPE915Fce7pDnIhCP8PoBwaw8U5JlTB"
}
